In [64]:
# importando bibliotecas
from faker import Faker
from random import choice, choices, randint, triangular, uniform
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from unicodedata import normalize
from re import sub, IGNORECASE

In [65]:
# Carregando variáveis de ambiente
load_dotenv()

# Acessando o banco de dados
DB_URL = os.getenv("DB_URL")
engine = create_engine(DB_URL)
faker = Faker('pt_BR')

In [66]:
def apenas_numeros(valor: str) -> str:
    return sub(r'\D', '', valor)

def remove_acentos(texto: str) -> str:
    return normalize('NFKD', texto.lower()).encode('ASCII', 'ignore').decode('ASCII')

In [67]:
# Constantes
QTD_LOJAS               = 10
QTD_BANCOS              = 5
QTD_CONTAS              = 80
QTD_PAGAMENTOS          = 1000
QTD_VENDAS              = 1000
QTD_TRANSPORTADORAS     = 8
QTD_FUNCIONARIOS        = 50
QTD_ENVIA_ITENS         = 3000
QTD_CONTA_LOJA          = 15
QTD_CONTA_FUNCIONARIO   = 60
QTD_PAGAMENTO_PARCELADO = 300

In [68]:
dados_loja = []
for i in range(QTD_LOJAS):
    cnpj_formatado = apenas_numeros(faker.cnpj())
    telefone_formatado = apenas_numeros(faker.cellphone_number())
    
    dados_loja.append({
        'nome': faker.company(),
        'cnpj': cnpj_formatado,
        'email': faker.email(),
        'telefone': telefone_formatado,
        'percentual_comissao': round(uniform(0.01, 0.2), 2)
    })

df_lojas = pd.DataFrame(dados_loja)
display(df_lojas.head())

,nome,cnpj,email,telefone,percentual_comissao
0,Carvalho,02671538000153,pedrocorreia@example.org,5597960010122,0.06
1,da Luz Castro e Filhos,74306129000169,antonella73@example.org,5558917443014,0.06
2,Araújo,83246951000172,calebe57@example.org,5565944080971,0.10
3,Costa S/A,23796541000173,vieiramanuella@example.net,55038972906305,0.07
4,Aragão Ltda.,65834927000172,mouraeduardo@example.org,5536984766523,0.07


In [69]:
dados_banco = []
for i in range(QTD_BANCOS):
    cnpj_formatado = apenas_numeros(faker.cnpj())
    
    dados_banco.append({
        'nome': f'Banco {faker.company()}',
        'cnpj': cnpj_formatado
    })

df_bancos = pd.DataFrame(dados_banco)
display(df_bancos.head())

,nome,cnpj
0,Banco Campos,92613840000148
1,Banco Aragão,31856492000145
2,Banco Borges,59807143000152
3,Banco Rios,12347905000111
4,Banco Rocha Ltda.,79658043000119


In [70]:
dados_conta = []

TIPOS_CONTA = ['CORRENTE', 'POUPANCA', 'SALARIO']

for i in range(QTD_CONTAS):
    dados_conta.append({
        'agencia': str(randint(1, 9999)).zfill(4),
        'numero_conta': str(randint(1, 999999)).zfill(6),
        'tipo_conta': choice(TIPOS_CONTA),
        'saldo': round(triangular(0, 9999.99, 100), 2),
        'cod_banco': randint(1, QTD_BANCOS)
    })

df_contas = pd.DataFrame(dados_conta)
display(df_contas.head())

,agencia,numero_conta,tipo_conta,saldo,cod_banco
0,8064,127863,CORRENTE,3756.50,2
1,1731,936164,SALARIO,1972.60,3
2,6887,359390,SALARIO,2416.36,2
3,4450,594697,CORRENTE,2583.62,1
4,0906,620877,POUPANCA,8788.85,4


In [71]:
dados_pagamento = []

TIPOS_PAGAMENTO = ['BOLETO', 'PIX', 'DEBITO', 'CREDITO']
STATUS_PAGAMENTO = ['AGUARDANDO', 'CONCLUIDO', 'EXPIRADO']

for i in range(QTD_PAGAMENTOS):
    cod_conta_origem = randint(1, QTD_CONTAS)
    cod_conta_destino = randint(1, QTD_CONTAS)
    while cod_conta_destino == cod_conta_origem:
        cod_conta_destino = randint(1, QTD_CONTAS)
    
    dados_pagamento.append({
        'tipo_pagamento': choice(TIPOS_PAGAMENTO),
        'status_pagamento': choice(STATUS_PAGAMENTO),
        'valor': round(triangular(0, 999.99, 100), 2),
        'cod_conta_origem': cod_conta_origem,
        'cod_conta_destino': cod_conta_destino
    })

df_pagamentos = pd.DataFrame(dados_pagamento)
display(df_pagamentos.head())

,tipo_pagamento,status_pagamento,valor,cod_conta_origem,cod_conta_destino
0,PIX,CONCLUIDO,233.93,9,49
1,DEBITO,CONCLUIDO,381.96,40,69
2,PIX,AGUARDANDO,234.12,20,11
3,PIX,AGUARDANDO,414.66,40,30
4,DEBITO,AGUARDANDO,128.63,63,75


In [72]:
dados_funcionario = []
for i in range(QTD_FUNCIONARIOS):
    nome = sub(r'\b(sr|sra|srta|dr|dra)\.\s*', '', faker.name(), flags=IGNORECASE)
    cpf_formatado = apenas_numeros(faker.cpf())
    telefone_formatado = apenas_numeros(faker.cellphone_number())
    email = f'{sub(r' ', '_', remove_acentos(nome))}@gmail.com'
    
    dados_funcionario.append({
        'nome': nome,
        'cpf': cpf_formatado,
        'cargo': faker.job(),
        'salario': round(triangular(1621, 4000, 100), 2),
        'email': email,
        'telefone': telefone_formatado,
        'cod_loja': randint(1, QTD_LOJAS)
    })

df_funcionarios = pd.DataFrame(dados_funcionario)
display(df_funcionarios.head())

,nome,cpf,cargo,salario,email,telefone,cod_loja
0,Catarina Pinto,15862734937,Coloproctologista,2610.05,catarina_pinto@gmail.com,5541911253375,8
1,Ravy Barbosa,72034598610,Capataz,2846.58,ravy_barbosa@gmail.com,55012925080125,5
2,André Gonçalves,15378924600,Profissional de relações internacionais,1966.55,andre_goncalves@gmail.com,5580950494630,1
3,Jade Correia,87592430150,Chapeiro,1942.10,jade_correia@gmail.com,5539919494786,9
4,Isadora Ferreira,70129845388,Radialista,2485.23,isadora_ferreira@gmail.com,55032963099539,3


In [73]:
dados_venda = []

STATUS_VENDA = ['APROVADO', 'CANCELADO', 'ESTORNADO']

for i in range(QTD_VENDAS):
    cod_vendedor = randint(1, QTD_FUNCIONARIOS)
    cod_loja = df_funcionarios.loc[
        cod_vendedor - 1, 'cod_loja'
    ]
    
    dados_venda.append({
        'valor_total': round(triangular(0, 999.99, 100), 2),
        'status_venda': choices(STATUS_VENDA, weights=[85, 5, 10], k=1)[0],
        'cod_vendedor': cod_vendedor,
        'cod_loja': cod_loja
    })

df_vendas = pd.DataFrame(dados_venda)
display(df_vendas.head())

,valor_total,status_venda,cod_vendedor,cod_loja
0,109.52,APROVADO,10,7
1,44.40,APROVADO,9,10
2,351.76,CANCELADO,16,9
3,466.36,APROVADO,39,8
4,80.43,ESTORNADO,36,9


In [74]:
dados_transportadora = []
for i in range(QTD_TRANSPORTADORAS):
    cnpj_formatado = apenas_numeros(faker.cnpj())
    telefone_formatado = apenas_numeros(faker.cellphone_number())
    
    dados_transportadora.append({
        'nome': faker.company(),
        'cnpj': cnpj_formatado,
        'telefone': telefone_formatado,
        'email': faker.email(),
        'taxa_entrega': round(uniform(0.01, 1), 2)
    })

df_transportadoras = pd.DataFrame(dados_transportadora)
display(df_transportadoras.head())

,nome,cnpj,telefone,email,taxa_entrega
0,Souza,63975214000101,5540910717380,maria-laura45@example.com,0.67
1,Martins Ltda.,57903814000153,5517902394438,sophia26@example.org,0.79
2,Rezende,19687034000124,55076955219397,jadecunha@example.com,0.95
3,Azevedo Moraes e Filhos,82693074000115,5572913284313,pedro-henriquefonseca@example.net,0.36
4,Oliveira,25943186000143,5550905053896,ravy13@example.org,1.00


In [75]:
dados_envia_itens = []
dados_conta_loja = []
dados_conta_funcionario = []
dados_pagamento_parcelado = []

for i in range(QTD_ENVIA_ITENS):
    dados_envia_itens.append({
        'cod_loja': randint(1, QTD_LOJAS),
        'cod_transportadora': randint(1, QTD_TRANSPORTADORAS)
    })

for i in range(QTD_CONTA_LOJA):
    dados_conta_loja.append({
        'cod_loja': randint(1, QTD_LOJAS),
        'cod_conta': randint(1, QTD_CONTAS)
    })

for i in range(QTD_CONTA_FUNCIONARIO):
    dados_conta_funcionario.append({
        'cod_funcionario': randint(1, QTD_FUNCIONARIOS),
        'cod_conta': randint(1, QTD_CONTAS)
    })

for i in range(QTD_PAGAMENTO_PARCELADO):
    dados_pagamento_parcelado.append({
        'cod_pagamento': randint(1, QTD_PAGAMENTOS),
        'cod_venda': randint(1, QTD_VENDAS)
    })

df_envia_itens = pd.DataFrame(dados_envia_itens)
df_conta_loja = pd.DataFrame(dados_conta_loja)
df_conta_funcionario = pd.DataFrame(dados_conta_funcionario)
df_pagamento_parcelado = pd.DataFrame(dados_pagamento_parcelado)

In [76]:
# Inserindo no banco de dados
tabelas = {
    'tb_loja': df_lojas,
    'tb_banco': df_bancos,
    'tb_transportadora': df_transportadoras,
    'tb_funcionario': df_funcionarios,
    'tb_conta': df_contas,
    'tb_pagamento': df_pagamentos,
    'tb_venda': df_vendas,
    'tb_envia_itens': df_envia_itens,
    'tb_conta_loja': df_conta_loja,
    'tb_conta_funcionario': df_conta_funcionario,
    'tb_pagamento_parcelado': df_pagamento_parcelado
}

print('Iniciando conexão com o banco de dados')
for t, df in tabelas.items():
    try:
        df.to_sql(name=t, con=engine, if_exists='append', index=False)
    except Exception as e:
        print(f'Um erro no banco aconteceu')
        print(e)
print('Conexão feita com sucesso')

Iniciando conexão com o banco de dados
Conexão feita com sucesso
